### Clean nav_history.csv

In [2]:
import pandas as pd
nav_history_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\02_nav_history.csv'
nav_history_df = pd.read_csv(nav_history_path)

print(nav_history_df.head())

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [4]:
nav_history_df['date'] = pd.to_datetime(nav_history_df['date'], errors='coerce')
nav_history_df.sort_values(by=['amfi_code','date'], inplace=True)
nav_history_df.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [5]:
nav_history_df.isna().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [43]:
def forward_fill_nav(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['amfi_code','date'])
    df['nav'] = pd.to_numeric(df['nav'], errors='coerce')

    out = []
    for amfi, g in df.groupby('amfi_code'):
        g = g.set_index('date').sort_index()
        # create a daily calendar (includes weekends & holidays)
        idx = pd.date_range(g.index.min(), g.index.max(), freq='D')
        full = g.reindex(idx)
        full['amfi_code'] = amfi
        # forward-fill NAV across weekends/holidays
        full['nav'] = full['nav'].ffill()
        full = full.reset_index().rename(columns={'index':'date'})
        # drop rows still missing nav (e.g., before first known NAV)
        full = full.dropna(subset=['nav'])
        out.append(full[['amfi_code','date','nav']])

    return pd.concat(out, ignore_index=True)

nav_history_df = forward_fill_nav(nav_history_df)

nav_history_df.drop_duplicates(subset=['amfi_code','date'], inplace=True)
nav_history_df.drop(nav_history_df[nav_history_df['nav'] < 0].index, inplace=True)
print(nav_history_df.isna().sum())
nav_history_df.dropna(inplace=True)
nav_history_df.head(20)

amfi_code    0
date         0
nav          0
dtype: int64


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-08,515.1639
6,100016,2022-01-09,515.1639
7,100016,2022-01-10,510.7136
8,100016,2022-01-11,513.5542
9,100016,2022-01-12,512.3195


In [9]:
nav_history_df.to_csv(r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\nav_history_cleaned.csv', index=False)

### Clean investor_transactions.csv

In [11]:
investor_transactions_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\08_investor_transactions.csv'
investor_transactions_df = pd.read_csv(investor_transactions_path)

investor_transactions_df.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [15]:
investor_transactions_df['transaction_date'] = pd.to_datetime(investor_transactions_df['transaction_date'], errors='coerce')
investor_transactions_df.sort_values(by=['transaction_date'], inplace=True)
print(investor_transactions_df['transaction_type'].value_counts())
investor_transactions_df.drop(investor_transactions_df[investor_transactions_df['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption']) == False].index, inplace=True)
investor_transactions_df.drop(investor_transactions_df[investor_transactions_df['amount_inr'] <= 0].index, inplace=True)
investor_transactions_df['kyc_status'].value_counts()

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [16]:
investor_transactions_df.to_csv(r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\investor_transactions_cleaned.csv', index=False)

### Clean scheme_performance.csv

In [17]:
scheme_performance_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\07_scheme_performance.csv'
scheme_performance_df = pd.read_csv(scheme_performance_path)

scheme_performance_df.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [18]:
scheme_performance_df.dtypes.iloc[:]


amfi_code               int64
scheme_name               str
fund_house                str
category                  str
plan                      str
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade                str
dtype: object

In [24]:
def get_anomalous_amfi_codes(df, col):
    numeric = pd.to_numeric(df[col], errors='coerce')
    q1 = numeric.quantile(0.25)
    q3 = numeric.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    anomaly_mask = (numeric < lower) | (numeric > upper)
    return df.loc[anomaly_mask, 'amfi_code'].dropna().unique().tolist()

anomally_amfi_list = []
for col in ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']:
    anomalous_amfi_codes = get_anomalous_amfi_codes(scheme_performance_df, col)
    anomally_amfi_list.append(anomalous_amfi_codes)
anomalous_amfi_codes_set = set().union(*anomally_amfi_list)
print(f"Anomalous AMFI Codes: {anomalous_amfi_codes_set}")


Anomalous AMFI Codes: {120844, 118636, 119598, 119599, 119120, 101207, 101208}


In [28]:
scheme_performance_df['anomally_flag'] = scheme_performance_df['amfi_code'].apply(lambda x: 1 if x in anomalous_amfi_codes_set else 0)

print(scheme_performance_df.shape)
scheme_performance_df.head()

(40, 20)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,anomally_flag
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,0
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,0
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,1
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,1
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,1


In [37]:
scheme_performance_df.drop(((scheme_performance_df[scheme_performance_df['expense_ratio_pct'] < 0.1])).index, inplace=True)
scheme_performance_df.drop(((scheme_performance_df[scheme_performance_df['expense_ratio_pct'] > 2.5 ])).index, inplace=True)
print(scheme_performance_df.shape)
scheme_performance_df.head()

(40, 20)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,anomally_flag
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,0
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,0
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,1
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,1
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,1


In [38]:
scheme_performance_df.to_csv(r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\scheme_performance_cleaned.csv', index=False)

### Basic cleaning

In [39]:

def basic_clean(df,name):
        df = df.copy()
        # strip column names and whitespace
        df.columns = [c.strip() for c in df.columns]
        for c in df.select_dtypes(include=['object']).columns:
            df[c] = df[c].astype(str).str.strip()
        df.to_csv(rf'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\{name}_cleaned.csv', index=False)

In [42]:
fund_master_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\01_fund_master.csv'
fund_master_df = pd.read_csv(fund_master_path)
basic_clean(fund_master_df,'fund_master')

aum_by_fund_house_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\03_aum_by_fund_house.csv'
aum_by_fund_house_df = pd.read_csv(aum_by_fund_house_path)
basic_clean(aum_by_fund_house_df,'aum_by_fund_house')

monthly_sip_inflows_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\04_monthly_sip_inflows.csv'
monthly_sip_inflows_df = pd.read_csv(monthly_sip_inflows_path)
basic_clean(monthly_sip_inflows_df,'monthly_sip_inflows')

category_inflows_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\05_category_inflows.csv'
category_inflows_df = pd.read_csv(category_inflows_path)
basic_clean(category_inflows_df,'category_inflows')

industry_folio_count_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\06_industry_folio_count.csv'
industry_folio_count_df = pd.read_csv(industry_folio_count_path)
basic_clean(industry_folio_count_df,'industry_folio_count')

portfolio_holdings_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\09_portfolio_holdings.csv'
portfolio_holdings_df = pd.read_csv(portfolio_holdings_path)
basic_clean(portfolio_holdings_df,'portfolio_holdings')

benchmark_indices_path = r'C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\data\raw\10_benchmark_indices.csv'
benchmark_indices_df = pd.read_csv(benchmark_indices_path)
basic_clean(benchmark_indices_df,'benchmark_indices')

C:\Users\Asus\AppData\Local\Temp\ipykernel_25052\3576056028.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=['object']).columns:
C:\Users\Asus\AppData\Local\Temp\ipykernel_25052\3576056028.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/mi